
# Interpretability Seminar

This notebook has two parts:

1. **Attention head visualisation** – we load a GPT‑2 model and visualise its attention patterns.  The goal is to identify phenomena such as attention sinks, copy/induction heads, and positional biases.
2. **Causal interventions and feature steering** – we perform activation patching on the Indirect Object Identification (IOI) task to locate where the model resolves pronoun reference, and we demonstrate simple feature steering through head ablation.

In [ ]:
!pip install -q torch matplotlib transformer_lens transformers circuitsvis

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
import transformers

import IPython.display as ipd
import circuitsvis as cv

model = HookedTransformer.from_pretrained("gpt2")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

In [ ]:
text = "When Mary and John went to the store, John gave a drink to"
model.generate(text)

In [ ]:
def visualise_attention(layer):
    assert layer in list(range(12))
    _, cache = model.run_with_cache(tokens)
    pattern = cache[f"blocks.{layer}.attn.hook_pattern"] # (batch, heads, Q_len, K_len)
    str_tokens = model.to_str_tokens(tokens)
    a = cv.attention.attention_patterns(tokens=str_tokens, attention=pattern[0])
    return a


text = "When Mary and John went to the store, John gave a drink to Mary."
tokens = model.to_tokens(text).to(device)

layer = 0
ipd.display(visualise_attention(layer));

### Attention heads (1 point)
Your task is to find the following attention heads (and please plot corresponding attention pattern):
- The earliest that attend to first Mary while generating second *
- Attention sink head
- Positional-bias head
- Induction head
- Punctuation-tracking head

In [ ]:
for layer in range(model.cfg.n_layers):
    # average attention weight on token 0 across heads and positions
    <YOUR CODE>
    print(f"Layer {layer}: avg attention on first token = {first_tok_attn:.2f}")



## Activation Patching and Feature Steering (2 points total)

Activation patching is *causal interventions* technique: run the model on a clean input (which exhibits correct behaviour) and a corrupted input (which exhibits incorrect behaviour), then replace a component of the corrupted run with the corresponding activation from the clean run.  If the model’s output is “fixed” by the patch, the replaced component is causal for the behaviour.

In [71]:
# IOI prompts
ioi_clean = "After John and Mary went to the store, Mary gave a bottle of milk to"
ioi_corrupt = "After John and Mary went to the store, John gave a bottle of milk to"
print(torch.where(model.to_tokens(ioi_clean) != model.to_tokens(ioi_corrupt))[1])

tensor([10], device='cuda:0')


In [ ]:
clean_tokens = model.to_tokens(ioi_clean).to(device)
corrupt_tokens = model.to_tokens(ioi_corrupt).to(device)

# Utility to compute logit difference between " John" and " Mary"
def logits_to_logit_diff(logits):
    john_id = model.to_single_token(" John")
    mary_id = model.to_single_token(" Mary")
    final_logits = logits[0, -1, :]
    return float(final_logits[john_id] - final_logits[mary_id])

# Baseline predictions
clean_logits = model(clean_tokens)
corrupt_logits = model(corrupt_tokens)
print("Clean prompt logit diff (John vs Mary):", logits_to_logit_diff(clean_logits))
print("Corrupt prompt logit diff (John vs Mary):", logits_to_logit_diff(corrupt_logits))


Clean prompt logit diff (John vs Mary): 4.276437759399414
Corrupt prompt logit diff (John vs Mary): -2.737588882446289


In [ ]:
_, clean_cache = model.run_with_cache(clean_tokens)
_, corrupt_cache = model.run_with_cache(corrupt_tokens)

layers = model.cfg.n_layers
logit_diff_effect = torch.zeros((layers, 1))

pos = 10
for L in range(layers):

    # Hook function to replace the residual stream at (layer L)
    def patch_resid_hook(resid, hook, position=pos, layer=L):
        <YOUR CODE>
        return resid

    # Run corrupted prompt with the hook
    patched_logits = model.run_with_hooks(
        clean_tokens,
        fwd_hooks=[(f"blocks.{L}.hook_resid_pre", patch_resid_hook)]
    )

    # Measure logit difference after patching
    logit_diff_effect[L] = logits_to_logit_diff(patched_logits)

plt.figure(figsize=(8, 3))
plt.scatter(range(12), logit_diff_effect.squeeze().numpy(), cmap='RdBu', c=logit_diff_effect)
plt.colorbar(label="Logit('John') − Logit('Mary')")
plt.xlabel("Layer")
plt.title("logit difference after activation patching")
plt.show()



### Feature steering via head ablation (1 point)

Once we have identified a head or layer that contributes to a behaviour, we can *steer* the model by intervening on it.  For example, ablation removes a head’s influence by zeroing its output.  Conversely, we could amplify a head’s output to bias the model.


In [ ]:
def generate_with_hooks(model: HookedTransformer, prompt: str, max_new_tokens: int, hooks)-> str: 
    tokens = model.to_tokens(prompt)
    tokens = tokens.to(model.cfg.device)

    for _ in range(max_new_tokens):
        logits = model.run_with_hooks(
            tokens,
            fwd_hooks=hooks
        )
        next_token = logits[0, -1].argmax().unsqueeze(0)
        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

        if next_token == model.tokenizer.eos_token_id:
            break

    return model.to_string(tokens[0, 1:])

def ablate_heads(model: HookedTransformer, heads, prompt: str, max_new_tokens: int = 20)-> str:
    by_layer = {}
    for layer, head_idx in heads:
        by_layer.setdefault(layer, []).append(head_idx)
    hooks = []
    for layer, head_list in by_layer.items():

        def make_hook(head_list=head_list):
            def hook(v, hook):
                # zeroing outputs of all hooked heads
                <YOUR CODE>
                return v
            return hook

        hooks.append((f"blocks.{layer}.attn.hook_v", make_hook()))

    return generate_with_hooks(
        model,
        prompt,
        max_new_tokens=max_new_tokens,
        hooks=hooks
    )


def scale_heads(model: HookedTransformer, heads, scale, prompt: str, max_new_tokens: int = 20)-> str:
    by_layer = {}
    for layer, head_idx in heads:
        by_layer.setdefault(layer, []).append(head_idx)
    hooks = []
    for layer, head_list in by_layer.items():

        def make_hook(head_list=head_list):
            def hook(v, hook):
                # scaling outputs of all hooked heads by scale
                <YOUR CODE>
                return v
            return hook

        hooks.append((f"blocks.{layer}.attn.hook_v", make_hook()))

    return generate_with_hooks(
        model,
        prompt,
        max_new_tokens=max_new_tokens,
        hooks=hooks
    )

In [ ]:

ioi_prompt = "After John and Mary went to the store, Mary gave a bottle of milk to"
max_new_tokens = 10

NAME_MOVER = [(9,6), (9,9), (10,0)]
NEG_NM     = [(10,7), (11,10)]

print(generate_completion(ioi_prompt, max_new_tokens=max_new_tokens))
print(ablate_heads(model, NAME_MOVER, ioi_prompt, max_new_tokens=max_new_tokens))
print(scale_heads(model, NEG_NM, 7.0, ioi_prompt, max_new_tokens=max_new_tokens))

### You may want to experiment here a bit and find more robust setup (for bonus points)
Here are papers that might help you: \
https://arxiv.org/pdf/2211.00593 \
https://arxiv.org/pdf/2310.04625 
